In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
gym_data_path = os.path.join("../../data/gym_data")

In [3]:
# Create list of filenames
filename_list = os.listdir(path=gym_data_path)

In [4]:
# Test for one file
testpath = os.path.join(gym_data_path, filename_list[118])
testopened = open(file=testpath, mode='r')
testread = testopened.read()
print(testread)

# leg front 1/24

Date: January 24, 2024

angled smith squat quad (35)☠️

9

6

6

leg extend (70)

6

death

machine should press front(40)

11

10

10

8

rear delt fly (50)

9

8


In [5]:
# Create list of lines from testread
test_split = testread.splitlines()

# Print date line as string
date_str = test_split[2]
date_sliced = date_str[6:]

# Convert to datetime
date = pd.to_datetime(date_sliced, format="%B %d, %Y")

In [6]:
# Create function to extract date from file
def extract_file_date(filename):
    # Get path of file
    path = os.path.join(gym_data_path, filename)
    
    opened = open(file=path, mode='r')
    
    read = opened.read()
    
    split = read.splitlines()
    
    date_str = split[2]
    date_sliced = date_str[6:]
    date = pd.to_datetime(date_sliced, format="%B %d, %Y")
    
    return date
    

In [7]:
# Test on first file in data/kys
extract_file_date('5 25 d326c23d60cf4e46bc594120503e0e55.md')

Timestamp('2023-05-25 00:00:00')

In [8]:
# Check all files for valid date lines
datetime_list = []
# IndexError suspicion: one of the files don't have a date
for i in filename_list:
    try:
        file_date = extract_file_date(i)
        
        datetime_list.append(file_date)
    except:
        print(f"Error in file: {i}")

Error in file: Untitled 27d3a7b1c7ab80db80d9c1c8ff9d663e.md
Error in file: Untitled 452615b6407b4babaea70da4f210c64b.md
Error in file: Untitled 21f3a7b1c7ab80fbbf59e16ba0db9a3d.md
Error in file: Pull 1 5bd444803d2e46e38c98bbea33ad0f91.md


In [9]:
gym_data = pd.Series(datetime_list, name='date').sort_values().reset_index()

In [10]:
# Create series of gym dates
gym_data.drop(labels='index', axis=1)

,date
0,2023-05-25
1,2023-05-27
2,2023-05-28
3,2023-05-29
4,2023-05-31
...,...
126,2025-10-17
127,2025-10-22
128,2025-10-29
129,2025-11-03


In [11]:
# Trying: iterate through split file, classify specific exercise into less specific exercise name
# Test for one file
testpath = os.path.join(gym_data_path, filename_list[118])
testopened = open(file=testpath, mode='r')
testread = testopened.read()
test_split = testread.splitlines()
test_split

['# leg front 1/24',
 '',
 'Date: January 24, 2024',
 '',
 'angled smith squat quad (35)☠️',
 '',
 '9',
 '',
 '6',
 '',
 '6',
 '',
 'leg extend (70)',
 '',
 '6',
 '',
 'death',
 '',
 'machine should press front(40)',
 '',
 '11',
 '',
 '10',
 '',
 '10',
 '',
 '8',
 '',
 'rear delt fly (50)',
 '',
 '9',
 '',
 '8']

In [12]:
# Test on one element if alphabetical
stripped = test_split[4].strip()
stripped[0].isalpha()

True

In [13]:
# Test on stripped if contains ()
if '(' in stripped and ')' in stripped:
    print('hi')

hi


In [14]:
# Create function to extract exercise lines
def extract_exercises(filename):
    # Get path, open, read, & split
    path = os.path.join(gym_data_path, filename)
    opened = open(file=path, mode='r')
    read = opened.read()
    split = read.splitlines()
    
    exercise_list = []
    
    for i in split:
        
        stripped = i.strip()
        
        if '(' in stripped and ')' in stripped:
            exercise_list.append(i)
            
    exercise_series = pd.Series(exercise_list).rename(index='exercise')
    return exercise_series

In [15]:
# Test function on one file
extract_exercises('upper a 5 27 2003a7b1c7ab80a7a6d8deaa62f0a9aa.md')

0                    inc (30)
1                lat pd (110)
2                    fly (90)
3        front delt mach (60)
4                   tbar (70)
5               preacher (15)
6                tri ext (10)
7    rear delt (16) diff mach
8    side delt (11) diff mach
Name: exercise, dtype: object

Try next: get dates assigned to exercises in dataframe

In [16]:
d = {'date': pd.Series(extract_file_date('upper a 5 27 2003a7b1c7ab80a7a6d8deaa62f0a9aa.md')),
     'exercise':extract_exercises('upper a 5 27 2003a7b1c7ab80a7a6d8deaa62f0a9aa.md')}
pd.DataFrame(data=d)

,date,exercise
0,2025-05-27,inc (30)
1,NaT,lat pd (110)
2,NaT,fly (90)
3,NaT,front delt mach (60)
4,NaT,tbar (70)
5,NaT,preacher (15)
6,NaT,tri ext (10)
7,NaT,rear delt (16) diff mach
8,NaT,side delt (11) diff mach


Trying: combine extract date & extract exercises into one function

- Returns filled dataframe of one file

In [17]:
def df_date_exercises(filename):
    # Get path, open, read, & split
    path = os.path.join(gym_data_path, filename)
    opened = open(file=path, mode='r')
    read = opened.read()
    split = read.splitlines()
    
    # Extract exercises first to get length
    exercise_list = []
    
    for i in split:
        
        stripped = i.strip()
        
        if '(' in stripped and ')' in stripped:
            exercise_list.append(i)
            
    exercise_series = pd.Series(exercise_list).rename(index='exercise')
    
    # Extract date of file
    try:
        date_str = split[2]
        date_sliced = date_str[6:]
        date = pd.to_datetime(date_sliced, format="%B %d, %Y")
        
        # Fill date series to match the length of exercise_series
        # Using pd.Series([date] * len(exercise_series)) ensures the series has the same length
        date_series = pd.Series([date] * len(exercise_series), name='date')

        # Build and return a DataFrame with the date repeated for each exercise
        df = pd.DataFrame({'date': date_series, 'exercise': exercise_series.reset_index(drop=True)})
        return df
    except:
        print("Error Reading File Date")
    


In [18]:
# Test on one file
df_date_exercises('upper a 8 3 2443a7b1c7ab80949c76c31a1ac44501.md')

,date,exercise
0,2025-08-03,lat pd (120)
1,2025-08-03,fly (85)
2,2025-08-03,tbar (i no like)
3,2025-08-03,shoulder press (35)


Trying: from our exercise column above, extract the weight in-between the parentheses

In [19]:
# Test regex extracting on one string
string_test = 'angled smith squat quad (35)☠️'

m = re.search(r'\((\d+)\)', string_test)
weight = int(m.group(1)) if m else None
print(weight)  

35


In [20]:
# Try to build independent function to extract weight
def extract_weight(filename):
    # Get path, open, read, & split
    path = os.path.join(gym_data_path, filename)
    opened = open(file=path, mode='r')
    read = opened.read()
    split = read.splitlines()
    
    weight_list = []
    
    for i in split:
    
        stripped = i.strip()
        
        if '(' in stripped and ')' in stripped:
            m = re.search(r'\((\d+)\)', stripped)
            weight = int(m.group(1)) if m else None
            weight_list.append(weight)
    
    weight_series = pd.Series(weight_list).rename(index='weight')
    
    return weight_series
            


In [21]:
extract_weight('upper a 8 3 2443a7b1c7ab80949c76c31a1ac44501.md')

0    120.0
1     85.0
2      NaN
3     35.0
Name: weight, dtype: float64

Trying: Add weight column to combined dataframe

In [22]:
def df_weight(filename):
    # Get path, open, read, & split
    path = os.path.join(gym_data_path, filename)
    opened = open(file=path, mode='r')
    read = opened.read()
    split = read.splitlines()
    
    # Extract exercises & weight first to get length
    exercise_list = []
    weight_list = []

    for i in split:
        stripped = i.strip()
        
        if '(' in stripped and ')' in stripped:
            # Append entire exercise string
            exercise_list.append(i)
            
            # Extract weight in-between parenthesis
            m = re.search(r'\((\d+)\)', stripped)
            weight = int(m.group(1)) if m else None
            weight_list.append(weight)
            
    exercise_series = pd.Series(exercise_list).rename(index='exercise')
    weight_series = pd.Series(weight_list).rename(index='weight')
    
    # Extract date of file
    try:
        date_str = split[2]
        date_sliced = date_str[6:]
        date = pd.to_datetime(date_sliced, format="%B %d, %Y")
        
        # Fill date series to match the length of exercise_series
        # Using pd.Series([date] * len(exercise_series)) ensures the series has the same length
        date_series = pd.Series([date] * len(exercise_series), name='date')

        # Build and return a DataFrame with the date repeated for each exercise
        df = pd.DataFrame({'date': date_series, 
                           'exercise': exercise_series.reset_index(drop=True),
                           'weight': weight_series.reset_index(drop=True)})
        return df
    except:
        print("Error Reading File Date")
    


In [23]:
df_weight('upper a 8 3 2443a7b1c7ab80949c76c31a1ac44501.md')

,date,exercise,weight
0,2025-08-03,lat pd (120),120.0
1,2025-08-03,fly (85),85.0
2,2025-08-03,tbar (i no like),NaN
3,2025-08-03,shoulder press (35),35.0


Trying: Extract number of reps for each exercise 

In [24]:
def extract_sets(filename):
    path = os.path.join(gym_data_path, filename)
    opened = open(file=path, mode='r')
    read = opened.read()
    split = read.splitlines()

    results = {}

    for index, i in enumerate(split):
        stripped = i.strip()

        # detect exercise lines like "machine fly (80)"
        if '(' in stripped and ')' in stripped:
            sets = []  # list to store all set numbers

            # look ahead through subsequent lines
            for next_line in split[index + 1:]:
                next_line = next_line.strip()

                if not next_line:  # skip blank lines
                    continue

                if next_line[0].isdigit():  # starts with a digit → a set number
                    try:
                        sets.append(int(next_line))  # convert to int if possible
                    except ValueError:
                        sets.append(next_line)  # if not pure number, keep as string
                    continue

                # if we hit another exercise before finding more sets, stop
                if '(' in next_line and ')' in next_line:
                    break

            # if no sets found, store None instead of empty list
            results[stripped] = sets if sets else None

    # print results nicely
    # for exercise, sets in results.items():
    #     print(f"{exercise} -> {sets}")
    
    # print(results.items())
    
    df = pd.DataFrame(results).melt(var_name="exercise", value_name="reps")
    return df


In [25]:
extract_sets('upper a 8 3 2443a7b1c7ab80949c76c31a1ac44501.md')

,exercise,reps
0,lat pd (120),10
1,lat pd (120),8
2,fly (85),10
3,fly (85),8
4,tbar (i no like),None
5,tbar (i no like),None
6,shoulder press (35),10
7,shoulder press (35),7


Trying: get exercises & sets into a dataframe

In [26]:
def extract_exercise_allinfo(filename):
    path = os.path.join(gym_data_path, filename)
    with open(path, "r") as opened:
        split = opened.read().splitlines()
        
    # --- extract date from the 3rd line ---
    try:
        date_str = split[2]
        date_sliced = date_str[6:]  # adjust slicing as needed
        date = pd.to_datetime(date_sliced, format="%B %d, %Y")
    except Exception:
        date = None  # fallback if parsing fails

    rows = []   # will store: {"exercise": ..., "weight": ..., "reps": ...}

    for index, i in enumerate(split):
        stripped = i.strip()

        # detect exercise lines like "machine fly (80)"
        if '(' in stripped and ')' in stripped:

            # --- extract weight ---
            m = re.search(r'\((\d+)\)', stripped)
            weight = int(m.group(1)) if m else None
            
            # --- clean exercise name ---
            exercise_name = re.sub(r'\s*\(\d+\)', '', stripped)

            # collect sets
            sets = []
            for next_line in split[index + 1:]:
                next_line = next_line.strip()

                if not next_line:
                    continue

                if next_line[0].isdigit():
                    try:
                        sets.append(int(next_line))
                    except ValueError:
                        sets.append(next_line)
                    continue

                # stop when next exercise encountered
                if '(' in next_line and ')' in next_line:
                    break

            # if no sets, still create one row with reps=None
            if not sets:
                rows.append({
                    "date":date,
                    "exercise": exercise_name,
                    "weight": weight,
                    "reps": None
                })
            else:
                # normal case: one row per set
                for rep in sets:
                    rows.append({
                        "date":date,
                        "exercise": exercise_name,
                        "weight": weight,
                        "reps": rep
                    })

    df = pd.DataFrame(rows)
    return df


In [27]:
extract_exercise_allinfo('upper b 7 13 22f3a7b1c7ab80279f08e774a1a21e07.md')

,date,exercise,weight,reps
0,2025-07-13,inc,35.0,7
1,2025-07-13,inc,35.0,5
2,2025-07-13,unilat pd,12.0,7 9
3,2025-07-13,front delt mach,60.0,6
4,2025-07-13,preacher ez,40.0,6
5,2025-07-13,preacher ez,40.0,5
6,2025-07-13,kenta tri (2 down),NaN,12
7,2025-07-13,kenta tri (2 down),NaN,11


In [35]:
df_list = []
empty = []

for file in filename_list:
    df = extract_exercise_allinfo(file)
    if df.shape[0] == 0:
        empty.append(file)
    else:
        df_list.append(df)

print("Empty:", empty)
full_df = pd.concat(df_list, ignore_index=True) 

Empty: ['cardio 21d3a7b1c7ab804db437fc93c9ee3ead.md', 'Legs Glute c44d9f1e8b514e2c87fa563a1618f774.md', 'Untitled 27d3a7b1c7ab80db80d9c1c8ff9d663e.md', 'cardio 5 29 2023a7b1c7ab80eb8e86fc3bd40c4c71.md', 'Untitled 452615b6407b4babaea70da4f210c64b.md', 'upper a 10 22 2943a7b1c7ab807bbecef9608753d13f.md', 'Untitled 21f3a7b1c7ab80fbbf59e16ba0db9a3d.md', '7 4 Push 18f17e48bfba4149b31fd352e40ff6b2.md', 'Pull 1 5bd444803d2e46e38c98bbea33ad0f91.md']


/var/folders/th/h57j9xd51jz6jynj6ycrqdqm0000gn/T/ipykernel_26804/2617480483.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat(df_list, ignore_index=True)


In [36]:
full_df['exercise'].unique()

array(['incline press', 'flat smith bench', 'machine delt raise',
       'machine fly', 'front delt machine', 'tricep cable unitlat',
       'Incline bench', 'Machine Press', 'Fly', 'cable pulldown',
       'tbar wide', 'seated row wide', 'unilat pulldown (8down) up it',
       'bicep curl', 'free squat', 'flat lev press', 'machine row lats',
       'leg ext upstairs', 'side delt cable raise',
       'death machine narrow stance', 'tbar (90‼️)', 'calf raise',
       'cable pulldown + 11', 't bar upper -15',
       'machine unilat pulldown (6down) up 1 both',
       'cable unilat rear delt', 'machine lever row +5',
       'machine unilat preacher', 'inc', 'unilat pd', 'front delt mach',
       'preacher ez', 'kenta tri (2 down)', 'black death',
       'glute press went 100 (go higher still)', 'smith ass squat',
       'leg curl', 'calf raise up it next time', 'leg press', 'leg ext',
       'backshots', 'glute push', 'squat', 'ext', '(+25)',
       'incline machine -5', 'bench', 'bi curl